# 06. Dots on Hands and Faces — MediaPipe, and Making Rules from Dots

> MediaPipe is Google's set of real-time video AI tools. It puts **21 dots** on each hand and **478 dots** on a face.
> With just the dot coordinates, you can answer questions like "How many fingers?", "Pinch?", "Mouth open?" with **simple math**.

### What you will learn
1. How to use MediaPipe Tasks: options → create → `recognize` / `detect`
2. **Normalized coordinates** (0~1) vs. pixel coordinates, and how the hand dots are numbered
3. Making rules from dot coordinates: counting fingers · pinch · mouth open
4. (Note) MediaPipe does not run on OpenVINO. It uses its own engine (TFLite) and runs on the **CPU**

### Hand dot numbers
```
        8   12  16  20       tip
        7   11  15  19
        6   10  14  18
    4   5    9  13  17       base
   3
  2                          0 = wrist
 1                           1~4 thumb · 5~8 index · 9~12 middle · 13~16 ring · 17~20 pinky
0
```

## 1. Setup

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "../..")).resolve()       # program folder (lab.bat tells us)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # model folder
core = ov.Core()
print("Model folder:", MODELS)
print("Devices:", core.available_devices)

import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision

MP = MODELS / "mediapipe"
hands = vision.GestureRecognizer.create_from_options(vision.GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path=str(MP / "gesture_recognizer.task")), num_hands=2))
face = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(MP / "face_landmarker.task")), num_faces=1))

def mp_image(bgr):
    """MediaPipe takes an mp.Image in RGB."""
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

## 2. Photo — take it while showing your hand

In [ ]:
PHOTO = "sample.jpg"          # photo to use when there is no webcam (same folder as this notebook)

def show(bgr, width=720):
    """Show an OpenCV photo (BGR) inside the notebook."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """Take one photo from the webcam. If there is no webcam, use the PHOTO file."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # some webcams give dark images for the first few frames
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"No webcam and no {PHOTO} — put a photo file name in PHOTO"
        print("No webcam, so using the file:", PHOTO)
    return frame

frame = grab()
print("Photo size (height, width, channels):", frame.shape)
show(frame)

## 3. 21 hand dots + hand gestures
- `hand_landmarks[i]` in the result = the 21 dots of hand number i. Each dot's `x, y` is a **0~1 ratio** (of the photo width · height). `z` is the depth relative to the wrist.
- `gestures[i]` = the gesture name and its score. It already knows 7 gestures: `Thumb_Up` (thumbs up) · `Thumb_Down` · `Victory` (V sign) · `Open_Palm` (open hand) · `Closed_Fist` (fist) · `Pointing_Up` (index finger up) · `ILoveYou`.
- `handedness[i]` = left hand / right hand. **If the photo is mirrored like a selfie, you get the opposite.**

In [ ]:
FINGERS = [(0, 1, 2, 3, 4), (0, 5, 6, 7, 8), (5, 9, 10, 11, 12), (9, 13, 14, 15, 16), (13, 17, 18, 19, 20), (0, 17)]

res = hands.recognize(mp_image(frame))
h, w = frame.shape[:2]
view = frame.copy()
for i, lms in enumerate(res.hand_landmarks):
    pts = [(int(p.x * w), int(p.y * h)) for p in lms]          # 0~1 → pixels
    for chain in FINGERS:
        for a, b in zip(chain, chain[1:]):
            cv2.line(view, pts[a], pts[b], (255, 200, 0), 2)
    for k, (x, y) in enumerate(pts):
        cv2.circle(view, (x, y), 4, (0, 0, 255), -1)
        cv2.putText(view, str(k), (x + 4, y - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 0), 1)
    g = res.gestures[i][0] if res.gestures and res.gestures[i] else None
    side = res.handedness[i][0].category_name if res.handedness else "?"
    print(f"Hand {i}: {side} · gesture {g.category_name} ({g.score:.2f})" if g else f"Hand {i}: {side}")
print("Hands found:", len(res.hand_landmarks))
show(view)

## 4. Making a rule ① — counting open fingers
A finger is open if its tip is **above** its second joint. In photo coordinates, **y gets bigger as you go down**, so "above" = **smaller** y.
The thumb opens sideways, so we compare x instead — the thumb is open if the thumb tip (4) is further out from the palm than the thumb joint (3). We find the "outward" direction by looking at the wrist (0) and the pinky base (17).

In [ ]:
TIPS, JOINTS = [8, 12, 16, 20], [6, 10, 14, 18]

def count_fingers(lms):
    n = sum(lms[t].y < lms[j].y for t, j in zip(TIPS, JOINTS))       # index ~ pinky: compare y
    outward = 1 if lms[17].x < lms[0].x else -1                        # if the pinky is on the left, the thumb opens to the right
    n += (lms[4].x - lms[3].x) * outward > 0                           # thumb: compare x
    return int(n)

for i, lms in enumerate(res.hand_landmarks):
    print(f"Hand {i}: {count_fingers(lms)} fingers open")

## 5. Making a rule ② — pinch (distance between thumb tip and index tip)
If you measure the distance in pixels, it gets smaller when the hand moves away from the camera. **Divide by the hand size** (wrist ~ middle finger base) and you get a value that does not depend on distance.

In [ ]:
def dist(a, b):
    return float(np.hypot(a.x - b.x, a.y - b.y))

for i, lms in enumerate(res.hand_landmarks):
    ratio = dist(lms[4], lms[8]) / (dist(lms[0], lms[9]) + 1e-6)     # divide by hand size
    print(f"Hand {i}: thumb-index ratio {ratio:.2f} → {'pinch' if ratio < 0.25 else 'apart'}")

## 6. 478 face dots — mouth open
Face dots have fixed numbers for the eyes · eyebrows · lips · face outline. 468~477 are the iris (pupil) dots.
Take the distance between 13 (inner upper lip) and 14 (inner lower lip) and divide it by the face height (10 forehead ~ 152 chin). That tells you how open the mouth is.

In [ ]:
r = face.detect(mp_image(frame))
view = frame.copy()
for lms in r.face_landmarks:
    for p in lms:
        cv2.circle(view, (int(p.x * w), int(p.y * h)), 1, (0, 255, 120), -1)
    open_ratio = dist(lms[13], lms[14]) / (dist(lms[10], lms[152]) + 1e-6)
    print(f"{len(lms)} dots · mouth open {open_ratio:.3f} → {'open' if open_ratio > 0.05 else 'closed'}")
show(view)

## Common errors
| Symptom | Cause · fix |
|---|---|
| `RuntimeError ... model_asset_path` | The `.task` file is missing → check `models/mediapipe` |
| No hand found | The hand is too small, cut off, or too dark → come closer, add light |
| Left/right hand is swapped | The photo is mirrored (check for `cv2.flip(frame, 1)`) |
| The rule works for some people but not others | Adjust the thresholds (0.25, 0.05) by actually testing them |

## Summary
- MediaPipe gives you a list of dots in **0~1 ratio coordinates**. Multiply by the photo size to get pixels.
- Rules made from dots should use **ratios**, so they do not change with distance · size.
- If rules (thresholds set by a person) are not enough → use the dot coordinates as features and **train** a model, like in notebook 07.

## Try it
1. Keep the webcam open, repeat in a loop, and show the finger count on screen in real time.
2. Detect a "V sign (✌)" using only dot rules, without the gesture name.
   <details><summary>Hint</summary>

   Check that the index · middle fingers are open (tip y < joint y) and the ring · pinky fingers are folded (tip y > joint y).
   </details>
3. Detect closed eyes (left eye: top 159 · bottom 145).
4. Use 21 dots × (x, y, z) = 63 numbers as features, train the classifier from notebook 07, and make your own hand gestures.

---
# 🛠 Mini project — Finger Rock-Paper-Scissors

**What to build**: **3 rounds of rock-paper-scissors** against the computer. After a countdown, the webcam takes a photo of your hand and judges it by the number of open fingers.
0 = rock · 2 = scissors · 5 = paper (anything else means "try again").

**What you use**: `hands` · `mp_image` · `count_fingers` (from this notebook), `random`

In [ ]:
import random

HAND = {0: "rock", 1: "rock", 2: "scissors", 3: "scissors", 4: "paper", 5: "paper"}     # forgive small mistakes
WIN = {("scissors", "paper"), ("rock", "scissors"), ("paper", "rock")}                 # (winner, loser)
score = {"me": 0, "computer": 0}

for rnd in range(1, 4):
    for c in ("Rock", "paper", "scissors!"):
        print(c, end=" ", flush=True); time.sleep(0.7)
    shot = grab()
    r = hands.recognize(mp_image(shot))
    if not r.hand_landmarks:
        print(f"\nRound {rnd}: I can't see your hand — try again"); continue
    n = count_fingers(r.hand_landmarks[0])
    me, com = HAND.get(n), random.choice(["scissors", "rock", "paper"])
    if (me, com) in WIN:
        score["me"] += 1; res = "You win!"
    elif (com, me) in WIN:
        score["computer"] += 1; res = "You lose"
    else:
        res = "Draw"
    print(f"\nRound {rnd}: you {me} ({n} fingers) vs computer {com} → {res}")
    show(shot, 240)
print("Final:", score)

**Try more**
- Judge by the gesture name (`gestures`) instead, and compare which works better: gesture names or finger counts
- Make the computer remember your last hands and play the winning move more often
- Judge with your own classifier from 07 instead of the finger count from 06